### Imports

In [1]:
from snowflake.snowpark.functions import col, sum as sum_, max as max_, datediff, round as round_, year, month, when, lit, lag, avg, count, stddev, to_date, dayofweek, weekofyear, dayofmonth, quarter, last_day
from snowflake.snowpark import Window
import re
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# from dwh_tools.get_or_create_session import get_or_create_session
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
import os
import requests
import gzip
import json
# from dwh_tools.get_or_create_session import get_or_create_session
pd.set_option('display.max_columns', None)
import time
import pywin
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


In [2]:
def get_or_create_session(schema: str = None) -> Session:
    """
    Returns the active Snowpark session if running inside Snowflake,
    otherwise creates one locally using Azure OAuth (interactive login if needed).
 
    Parameters
    ----------
    config_path : str
        Path to a JSON config file containing Snowflake connection parameters
        (account, warehouse, role, database, schema).
 
    Returns
    -------
    Session
        A Snowpark Session object.
    """
    # If running on snowflake
    if 'POSIT_PRODUCT' in os.environ:
        session = Session.builder.getOrCreate()
        session.sql("USE DATABASE PROD_FOR_SKADE_PRODUKT_ADHOC").collect()
        if schema:
            session.sql("USE SCHEMA " + schema).collect()
        else:
            session.sql("USE SCHEMA PRODUKT_WRITE_DEV").collect()
        session.sql("USE WAREHOUSE SKADE_VWH").collect()
 
        return session
 
    try:
        session = get_active_session()
        return session
    except Exception:
        import win32api
        if schema is None:
            schema = 'PRODUKT_WRITE_DEV'
 
        connection_parameters = {
            "server": "km28161.west-europe.azure.snowflakecomputing.com",
            "warehouse": "SKADE_VWH",
            "account": "VK82539-KLP",
            "database": "PROD_FOR_SKADE_PRODUKT_ADHOC",
            "schema" : schema,
            "user": win32api.GetUserNameEx(win32api.NameUserPrincipal),  
            "authenticator": "externalbrowser"
        }
       
        # Create the session
        session = Session.builder.configs(connection_parameters).create()
        return session

In [3]:
session = get_or_create_session()

### Data

In [4]:
df_inngang = session.table('elh_write.inngangsdata').to_pandas()
df_info = session.table('inngangsdata_info').to_pandas()
df_weather = pd.read_csv("../../data/oslo_weather.csv", delimiter=";")

In [5]:
# Fjerner noen kolonner: Beholder kun 30 dager bak, fjerner fremover og 7 dager bak. 
cols_to_remove = [
    "hf_dato",
    "antall_nye_kunder_b7_mpb01_ny",
    "antall_nye_kunder_f30_mpb01_ny",
    "antall_nye_kunder_f7_mpb01_ny",
    "antall_hf_b7_mpb01_ny",
    "antall_hf_f30_mpb01_ny",
    "antall_hf_f7_mpb01_ny",
    "antall_nye_kunder_b7_eph01_for",
    "antall_nye_kunder_f30_eph01_for",
    "antall_nye_kunder_f7_eph01_for",
    "antall_hf_b7_eph01_for",
    "stddev_premieendring_b7_eph01_for",
    "snitt_premieendring_b7_eph01_for",
    "antall_hf_f30_eph01_for",
    "stddev_premieendring_f30_eph01_for",
    "snitt_premieendring_f30_eph01_for",
    "antall_hf_f7_eph01_for",
    "stddev_premieendring_f7_eph01_for",
    "snitt_premieendring_f7_eph01_for",
    "antall_nye_kunder_b7_eph01_ny",
    "antall_nye_kunder_f30_eph01_ny",
    "antall_nye_kunder_f7_eph01_ny",
    "antall_hf_b7_eph01_ny",
    "antall_hf_f30_eph01_ny",
    "antall_hf_f7_eph01_ny",
    "antall_nye_kunder_b7_epf01_ny",
    "antall_nye_kunder_f30_epf01_ny",
    "antall_nye_kunder_f7_epf01_ny",
    "antall_hf_b7_epf01_ny",
    "antall_hf_f30_epf01_ny",
    "antall_hf_f7_epf01_ny",
    "antall_nye_kunder_b7_epf01_for",
    "antall_nye_kunder_f30_epf01_for",
    "antall_nye_kunder_f7_epf01_for",
    "antall_hf_b7_epf01_for",
    "stddev_premieendring_b7_epf01_for",
    "snitt_premieendring_b7_epf01_for",
    "antall_hf_f30_epf01_for",
    "stddev_premieendring_f30_epf01_for",
    "snitt_premieendring_f30_epf01_for",
    "antall_hf_f7_epf01_for",
    "stddev_premieendring_f7_epf01_for",
    "snitt_premieendring_f7_epf01_for",
    "antall_nye_kunder_b7_upr01_ny",
    "antall_nye_kunder_f30_upr01_ny",
    "antall_nye_kunder_f7_upr01_ny",
    "antall_hf_b7_upr01_ny",
    "antall_hf_f30_upr01_ny",
    "antall_hf_f7_upr01_ny",
    "antall_nye_kunder_b7_mpb01_for",
    "antall_nye_kunder_f30_mpb01_for",
    "antall_nye_kunder_f7_mpb01_for",
    "antall_hf_b7_mpb01_for",
    "stddev_premieendring_b7_mpb01_for",
    "snitt_premieendring_b7_mpb01_for",
    "antall_hf_f30_mpb01_for",
    "stddev_premieendring_f30_mpb01_for",
    "snitt_premieendring_f30_mpb01_for",
    "antall_hf_f7_mpb01_for",
    "stddev_premieendring_f7_mpb01_for",
    "snitt_premieendring_f7_mpb01_for",
    "antall_nye_kunder_b7_upr01_for",
    "antall_nye_kunder_f30_upr01_for",
    "antall_nye_kunder_f7_upr01_for",
    "antall_hf_b7_upr01_for",
    "stddev_premieendring_b7_upr01_for",
    "snitt_premieendring_b7_upr01_for",
    "antall_hf_f30_upr01_for",
    "stddev_premieendring_f30_upr01_for",
    "snitt_premieendring_f30_upr01_for",
    "antall_hf_f7_upr01_for",
    "stddev_premieendring_f7_upr01_for",
    "snitt_premieendring_f7_upr01_for"
]

In [6]:
from pathlib import Path
import sys

project_code_dir = Path.cwd().resolve().parent
if str(project_code_dir) not in sys.path:
    sys.path.append(str(project_code_dir))

from utils.data_prep import data_prep

df = data_prep(df_inngang=df_inngang, df_info=df_info, df_weather=df_weather, cols_to_remove=cols_to_remove)

In [7]:
df.head()

,ankomst_dato,antall_samtaler,behandlet_andel,tid_i_ko_snitt,behandlingstid_snitt,total_behandlingstid,etterbehandligstid_snitt,total_etterbehandligstid,antall_nye_kunder_b30_mpb01_ny,antall_hf_b30_mpb01_ny,antall_nye_kunder_b30_eph01_for,antall_hf_b30_eph01_for,stddev_premieendring_b30_eph01_for,snitt_premieendring_b30_eph01_for,antall_nye_kunder_b30_eph01_ny,antall_hf_b30_eph01_ny,antall_nye_kunder_b30_epf01_ny,antall_hf_b30_epf01_ny,antall_nye_kunder_b30_epf01_for,antall_hf_b30_epf01_for,stddev_premieendring_b30_epf01_for,snitt_premieendring_b30_epf01_for,antall_nye_kunder_b30_upr01_ny,antall_hf_b30_upr01_ny,antall_nye_kunder_b30_mpb01_for,antall_hf_b30_mpb01_for,stddev_premieendring_b30_mpb01_for,snitt_premieendring_b30_mpb01_for,antall_nye_kunder_b30_upr01_for,antall_hf_b30_upr01_for,stddev_premieendring_b30_upr01_for,snitt_premieendring_b30_upr01_for,aar,kvartal,maaned,ukenummer,ukedag,dag_i_maaned,er_helg,helligdag,er_helligdag,er_dag_foer_helligdag,er_dag_etter_helligdag,navn,stasjon,tid(norsk normaltid),middeltemperatur,nedbør,middeltemp_mndsnitt,tempavvik_fra_mndsnitt,antall_nye_kunder_b30_ny,antall_nye_kunder_b30_for,antall_hf_b30_ny,antall_hf_b30_for,stddev_premieendring_b30_for,snitt_premieendring_b30_for,antall_nye_kunder_b30_tot,antall_hf_b30_tot
0,2024-06-03,181,0.850829,274.342541,281.049724,50870,130.419890,23606,501,2900,96.0,4921.0,7.780343,1.119475,96.0,961.0,13.0,176.0,13.0,988.0,5.220043,1.076235,105,610,501.0,6558.0,7.516885,1.185686,105.0,2338.0,1.935644,1.032337,2024,2,6,23,1,3,0,None,0,0,0,Oslo - Bygdøy Ii,SN18810,2024-06-03,19.0,0.0,15.72,3.28,715.0,715.0,4647.0,14805.0,5.613229,1.103433,1430.0,19452.0
1,2024-06-04,408,0.828431,732.632353,280.009804,114244,121.987745,49771,502,2907,89.0,4949.0,7.758312,1.121448,89.0,936.0,11.0,166.0,11.0,996.0,5.199079,1.076031,102,601,502.0,6586.0,7.459350,1.185899,102.0,2343.0,1.946560,1.032964,2024,2,6,23,2,4,0,None,0,0,0,Oslo - Bygdøy Ii,SN18810,2024-06-04,15.1,0.0,15.72,-0.62,704.0,704.0,4610.0,14874.0,5.590825,1.104086,1408.0,19484.0
2,2024-06-05,294,0.782313,650.248299,258.166667,75901,97.272109,28598,488,2880,85.0,4947.0,7.759890,1.121574,85.0,920.0,10.0,158.0,10.0,1023.0,5.130114,1.075870,104,606,488.0,6599.0,7.504770,1.186168,104.0,2356.0,1.941469,1.033941,2024,2,6,23,3,5,0,None,0,0,0,Oslo - Bygdøy Ii,SN18810,2024-06-05,13.1,11.4,15.72,-2.62,687.0,687.0,4564.0,14925.0,5.584061,1.104388,1374.0,19489.0
3,2024-06-06,313,0.808307,447.830671,294.220447,92091,91.447284,28623,474,2844,87.0,4975.0,7.738009,1.122787,87.0,925.0,9.0,155.0,9.0,1027.0,6.564680,1.081112,103,592,474.0,6627.0,7.799477,1.187572,103.0,2371.0,2.172918,1.034799,2024,2,6,23,4,6,0,None,0,0,0,Oslo - Bygdøy Ii,SN18810,2024-06-06,11.6,8.8,15.72,-4.12,673.0,673.0,4516.0,15000.0,6.068771,1.106567,1346.0,19516.0
4,2024-06-07,371,0.867925,791.533693,301.137466,111722,133.552561,49548,481,2890,87.0,4989.0,8.028814,1.123380,87.0,946.0,9.0,155.0,9.0,1024.0,6.574265,1.081179,105,603,481.0,6644.0,7.754416,1.187596,105.0,2373.0,2.172067,1.035438,2024,2,6,23,5,7,0,None,0,0,0,Oslo - Bygdøy Ii,SN18810,2024-06-07,11.0,1.0,15.72,-4.72,682.0,682.0,4594.0,15030.0,6.132390,1.106898,1364.0,19624.0
